# 03 · Treino do adaptador LoRA (Google Colab)

Este notebook treina o adaptador LoRA usado na configuração 3 (RAG + LoRA) sobre o modelo `Qwen/Qwen3-4B-Instruct-2507`. Os exemplos estão em `treino/treino.jsonl` e `treino/validacao.jsonl` e foram montados pela mesma função que monta o prompt da API (`src/luz_segura/prompt.py`). Assim, o treino e o uso real veem o mesmo formato, caractere por caractere.

**Como executar**

1. No Colab: Arquivo > Fazer upload de notebook.
2. Ambiente de execução > Alterar o tipo de ambiente de execução > **GPU T4**. Se aparecer o campo de versão do ambiente de execução, escolha **2026.10**.
3. Ambiente de execução > **Executar tudo**. O Colab pede os arquivos `treino.jsonl` e `validacao.jsonl` (pasta `treino/` do repositório) e, depois, acesso ao Google Drive (os checkpoints ficam lá para o treino continuar se a sessão cair).
4. No fim, o navegador baixa `adaptador_luz_segura.zip` (cerca de 200 MB; deixe-o fora do repositório, e uma cópia fica no Drive). Baixe também o notebook executado (Arquivo > Baixar > Baixar o .ipynb), salve-o em `notebooks/03_treino_lora_colab.ipynb`, no lugar deste, e tire os metadados da conta Google com `python treino/clean_colab_notebook.py notebooks/03_treino_lora_colab.ipynb`.

**Modo ensaio.** Com `MODE = "rehearsal"`, o notebook não pede arquivos: usa exemplos de mentira com contexto longo, treina duas épocas curtas, testa a reserva em float16 e gera um adaptador pequeno para conferir a conversão no PC. Leva cerca de 15 minutos.

| Arquivo no zip | Conteúdo |
|---|---|
| `epoch_1/`, `epoch_2/`, `epoch_3/` | adaptador PEFT de cada época, em float16 |
| `training_metrics.json` | perdas de treino e de validação, melhor época pela perda de validação, duração, memória de pico e versões |
| `rendered_example.txt` | primeiro exemplo de treino já no formato de chat do modelo |

In [1]:
# Configuração
import os

MODE = "train"        # "train" treina de verdade; "rehearsal" só testa o ambiente (cerca de 15 min, sem arquivos)
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# Permite rodar o notebook fora do Colab com outro modelo, para teste
MODE = os.environ.get("LUZ_MODE", MODE)
BASE_MODEL = os.environ.get("LUZ_BASE_MODEL", BASE_MODEL)

MAX_LENGTH = 2048     # limite de tokens por exemplo; o maior exemplo de treino tem de caber com folga
USE_DRIVE = True      # checkpoints no Google Drive (só no modo "train")
FORCE_FP16 = False    # True pula a quantização em 4 bits e treina com a base em float16
SEED = 42
REHEARSAL = MODE == "rehearsal"
assert MODE in ("train", "rehearsal")

In [2]:
# Ambiente: bibliotecas fixadas e GPU
import importlib.metadata as md
import json
import platform
import subprocess
import sys
import time

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.22.1", "peft==0.21.2", "trl==1.18.0", "bitsandbytes==0.50.2"], check=True)

import torch

HAS_GPU = torch.cuda.is_available()
if IN_COLAB:
    assert HAS_GPU, "Sessão sem GPU: Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4"
DEVICE_NAME = torch.cuda.get_device_properties(0).name if HAS_GPU else "CPU"
VERSIONS = {"python": platform.python_version(), "colab": os.environ.get("COLAB_RELEASE_TAG")}
for package in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "bitsandbytes"):
    try:
        VERSIONS[package] = md.version(package)
    except md.PackageNotFoundError:
        VERSIONS[package] = None
print("dispositivo:", DEVICE_NAME, f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB)" if HAS_GPU else "")
print(json.dumps(VERSIONS, indent=1))

dispositivo: Tesla T4 (14.6 GiB)
{
 "python": "3.13.16",
 "colab": "release-colab-external-images_20261116-060055_RC00",
 "torch": "2.14.1+cu130",
 "transformers": "5.22.1",
 "peft": "0.21.2",
 "trl": "1.18.0",
 "accelerate": "1.15.0",
 "datasets": "4.8.5",
 "bitsandbytes": "0.50.2"
}


In [3]:
# Dados: arquivos de treino e validação (ou exemplos de mentira no ensaio)
import hashlib
import random

from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
WORK_DIR = "/content" if IN_COLAB else os.path.abspath("saida_treino")
os.makedirs(WORK_DIR, exist_ok=True)


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def rehearsal_examples(count, seed):
    # Exemplos de mentira com contexto longo: medem memória e tempo, não ensinam nada.
    rng = random.Random(seed)
    sentence = ("A manutenção da iluminação pública depende do registro do pedido "
                "com o endereço completo e a referência do poste. ")
    examples = []
    for i in range(count):
        repeats = 22 if i == 0 else rng.randint(8, 18)  # o primeiro é o mais longo
        context = "\n\n".join(f"[T{j}] Documento de teste {j}\n" + sentence * repeats for j in (1, 2, 3))
        answerable = i % 3 != 0
        target = {"consta": answerable,
                  "resposta": "Registre o pedido informando o endereço e a referência do poste." if answerable else "",
                  "fontes": ["T1"] if answerable else []}
        examples.append({
            "prompt": [{"role": "system", "content": "Responda somente com um objeto JSON."},
                       {"role": "user", "content": f"Trechos:\n{context}\n\nPergunta: pergunta de teste {i}?"}],
            "completion": [{"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}],
        })
    return examples


if REHEARSAL:
    train_rows, val_rows = rehearsal_examples(8, 1), rehearsal_examples(4, 2)
else:
    files = {name: os.path.join(WORK_DIR, name) for name in ("treino.jsonl", "validacao.jsonl")}
    if IN_COLAB and not all(os.path.exists(p) for p in files.values()):
        from google.colab import files as colab_files
        print("Selecione treino.jsonl e validacao.jsonl (pasta treino/ do repositório).")
        os.chdir(WORK_DIR)
        colab_files.upload()
    missing = [name for name, path in files.items() if not os.path.exists(path)]
    assert not missing, f"Arquivos ausentes em {WORK_DIR}: {missing}"
    train_rows, val_rows = read_jsonl(files["treino.jsonl"]), read_jsonl(files["validacao.jsonl"])

DATA_HASH = hashlib.sha256(json.dumps([train_rows, val_rows], ensure_ascii=False).encode("utf-8")).hexdigest()


def count_tokens(example):
    text = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


TOKEN_STATS = {}
for name, rows in (("train", train_rows), ("validation", val_rows)):
    sizes = sorted(count_tokens(ex) for ex in rows)
    TOKEN_STATS[name] = {"examples": len(rows), "total": sum(sizes), "median": sizes[len(sizes) // 2], "max": sizes[-1]}
    assert sizes[-1] <= MAX_LENGTH - 8, f"há exemplo de {name} com {sizes[-1]} tokens: seria cortado"
    for ex in rows:
        target = json.loads(ex["completion"][0]["content"])
        assert list(target) == ["consta", "resposta", "fontes"], target
print(json.dumps(TOKEN_STATS, indent=1), "\nsha256 dos dados:", DATA_HASH)

RENDERED_EXAMPLE = tokenizer.apply_chat_template(train_rows[0]["prompt"] + train_rows[0]["completion"], tokenize=False)
print(RENDERED_EXAMPLE[-700:])
train_ds, val_ds = Dataset.from_list(train_rows), Dataset.from_list(val_rows)

{
 "train": {
  "examples": 328,
  "total": 345390,
  "median": 1037,
  "max": 1545
 },
 "validation": {
  "examples": 37,
  "total": 38629,
  "median": 982,
  "max": 1549
 }
} 
sha256 dos dados: 2012ca226b0dc8b02298c740577c1c425826840e90ab29facc8dac08d3aefddd
á executa o serviço, e o pedido do cidadão continua indo para a Central 156.

[T3] Luz Segura, tratamento da foto > Localização e tempo de guarda da foto
A localização enviada é arredondada para o trecho de rua, com precisão de cerca de 30 metros, antes de ser guardada. A foto fica guardada por até 90 dias ou até o ponto ser marcado como resolvido, o que acontecer primeiro; depois é apagada, e ficam só a classificação, o tipo de problema e a localização arredondada. As fotos não são publicadas: o mapa público mostra apenas pontos agregados.

Pergunta: posso podar a arvore q ta tapando o poste eu mesmo?<|im_end|>
<|im_start|>assistant
{"consta": false, "resposta": "", "fontes": []}<|im_end|>



In [4]:
# Pastas: checkpoints no Drive, numa pasta própria para cada conjunto de dados
import glob
import shutil

if IN_COLAB and USE_DRIVE and not REHEARSAL:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/luz_segura_treino"
else:
    PERSIST_DIR = os.path.join(WORK_DIR, "luz_segura_treino")
CHECKPOINT_DIR = os.path.join(PERSIST_DIR, ("rehearsal_" if REHEARSAL else "checkpoints_") + DATA_HASH[:10])
if REHEARSAL:
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
PREVIOUS_CHECKPOINTS = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))
print("checkpoints em", CHECKPOINT_DIR, "| retomando" if PREVIOUS_CHECKPOINTS else "| treino novo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoints em /content/drive/MyDrive/luz_segura_treino/checkpoints_2012ca226b | treino novo


In [5]:
# Treino: QLoRA em 4 bits, com reserva automática em float16
import gc

from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

LORA_CONFIG = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
EPOCHS = 2 if REHEARSAL else 3
GRAD_ACCUM = 2 if REHEARSAL else 8


def load_model(use_4bit):
    options = {"dtype": torch.float16 if HAS_GPU else torch.float32, "attn_implementation": "sdpa"}
    if HAS_GPU:
        options["device_map"] = {"": 0}
    if use_4bit:
        options["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL, **options)


def make_args(output_dir, epochs, grad_accum, max_steps=-1, evaluate=True):
    return SFTConfig(
        output_dir=output_dir, max_length=MAX_LENGTH,
        per_device_train_batch_size=1, gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs, max_steps=max_steps,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=2 if REHEARSAL else 10, weight_decay=0.01,
        fp16=HAS_GPU, bf16=False,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1 if REHEARSAL else 5,
        eval_strategy="epoch" if evaluate else "no", per_device_eval_batch_size=1,
        save_strategy="epoch" if evaluate else "no", save_total_limit=EPOCHS,
        load_best_model_at_end=evaluate, metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none", seed=SEED,
    )


USE_4BIT = HAS_GPU and not FORCE_FP16
try:
    model = load_model(USE_4BIT)
except Exception as error:
    if not USE_4BIT:
        raise
    print("A carga em 4 bits falhou; seguindo com a base em float16:", repr(error))
    USE_4BIT = False
    gc.collect()
    torch.cuda.empty_cache()
    model = load_model(False)

trainer = SFTTrainer(
    model=model, args=make_args(CHECKPOINT_DIR, EPOCHS, GRAD_ACCUM),
    train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer, peft_config=LORA_CONFIG,
)
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
trainer.model.print_trainable_parameters()
if HAS_GPU:
    torch.cuda.reset_peak_memory_stats()
start = time.time()
train_result = trainer.train(resume_from_checkpoint=True if PREVIOUS_CHECKPOINTS else None)
DURATION_S = time.time() - start
PEAK_MEMORY_GIB = torch.cuda.max_memory_allocated() / 2**30 if HAS_GPU else None
print(train_result)
print(f"duração {DURATION_S / 60:.1f} min | memória de pico {PEAK_MEMORY_GIB} GiB | 4 bits: {USE_4BIT}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/328 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/328 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/328 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/328 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/37 [00:00<?, ? examples/s]

... (2 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/37 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.210755,0.237048,0.151299,0.939541,345390.000000
2,0.130677,0.244262,0.105865,0.942122,690780.000000
3,0.076541,0.254485,0.091192,0.943724,1036170.000000


TrainOutput(global_step=123, training_loss=0.20056573925464133, metrics={'train_runtime': 2794.3508, 'train_samples_per_second': 0.352, 'train_steps_per_second': 0.044, 'total_flos': 2.27949654422016e+16, 'train_loss': 0.20056573925464133, 'epoch': 3.0})
duração 46.6 min | memória de pico 3.6069583892822266 GiB | 4 bits: True


In [6]:
# Conferência rápida: o modelo treinado responde em JSON e para sozinho?
final_model = trainer.model.eval()
final_model.config.use_cache = True
SANITY_CHECK = []
for ex in val_rows[:3]:
    text = tokenizer.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(final_model.device)
    with torch.no_grad():
        output = final_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        json.loads(generated)
        valid = True
    except json.JSONDecodeError:
        valid = False
    SANITY_CHECK.append({"generated": generated, "expected": ex["completion"][0]["content"], "valid_json": valid})
    print("JSON válido:", valid, "\n  gerado:  ", generated[:400], "\n  esperado:", ex["completion"][0]["content"][:400])

JSON válido: True 
  gerado:   {"consta": false, "resposta": "", "fontes": []} 
  esperado: {"consta": false, "resposta": "", "fontes": []}
JSON válido: True 
  gerado:   {"consta": true, "resposta": "Não. O portal da Central 156 informa prazo médio, e não prazo garantido. Ele avisa que o tempo de análise pode variar de acordo com a demanda.", "fontes": ["T1"]} 
  esperado: {"consta": true, "resposta": "O assistente não informa prazos que não estejam publicados.", "fontes": ["T3"]}
JSON válido: True 
  gerado:   {"consta": true, "resposta": "Sim. O art. 454, inciso V, da REN ANEEL 1.000/2021 diz que a distribuidora deve ter procedimentos para inspeção e correção de deficiência técnica ou de segurança que ofereçam risco de danos a pessoas, bens ou ao funcionamento do sistema elétrico ou de iluminação pública.", "fontes": ["T2"]} 
  esperado: {"consta": true, "resposta": "Sim. O art. 454, inciso V, da REN ANEEL 1.000/2021 diz que a norma técnica da distribuidora deve ter procedimentos pa

In [7]:
# Entrega: adaptador de cada época em float16, métricas e zip
from safetensors.torch import load_file, save_file

OUTPUT_DIR = os.path.join(WORK_DIR, "adaptador_luz_segura")
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR)
evals = [r for r in trainer.state.log_history if "eval_loss" in r]
epoch_by_step = {r["step"]: round(r["epoch"]) for r in evals}
checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")), key=lambda c: int(c.rsplit("-", 1)[1]))
for checkpoint in checkpoints:
    step = int(checkpoint.rsplit("-", 1)[1])
    folder = os.path.join(OUTPUT_DIR, f"epoch_{epoch_by_step.get(step, step)}")
    os.makedirs(folder)
    weights = load_file(os.path.join(checkpoint, "adapter_model.safetensors"))
    save_file({k: v.to(torch.float16).contiguous() for k, v in weights.items()}, os.path.join(folder, "adapter_model.safetensors"))
    shutil.copy(os.path.join(checkpoint, "adapter_config.json"), folder)
    print(os.path.basename(folder), "<-", os.path.basename(checkpoint), len(weights), "tensores")

best = trainer.state.best_model_checkpoint
METRICS = {
    "mode": MODE, "base_model": BASE_MODEL, "use_4bit": USE_4BIT, "device": DEVICE_NAME,
    "data_sha256": DATA_HASH, "tokens": TOKEN_STATS,
    "hyperparameters": {"r": 16, "lora_alpha": 16, "lora_dropout": 0.05, "learning_rate": 2e-4,
                        "scheduler": "cosine", "epochs": EPOCHS, "batch_size": 1, "grad_accum": GRAD_ACCUM,
                        "max_length": MAX_LENGTH, "seed": SEED},
    "best_checkpoint": os.path.basename(best) if best else None,
    "best_epoch": epoch_by_step.get(int(best.rsplit("-", 1)[1])) if best else None,
    "eval_loss_by_epoch": [{"epoch": round(r["epoch"]), "step": r["step"], "loss": r["eval_loss"]} for r in evals],
    "log_history": trainer.state.log_history,
    "duration_min": round(DURATION_S / 60, 1), "peak_memory_gib": PEAK_MEMORY_GIB,
    "sanity_check": SANITY_CHECK, "versions": VERSIONS,
}
with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(METRICS, f, ensure_ascii=False, indent=1)
with open(os.path.join(OUTPUT_DIR, "rendered_example.txt"), "w", encoding="utf-8") as f:
    f.write(RENDERED_EXAMPLE)
print(json.dumps(METRICS["eval_loss_by_epoch"], indent=1), "\nmelhor época:", METRICS["best_epoch"])

epoch_1 <- checkpoint-41 504 tensores
epoch_2 <- checkpoint-82 504 tensores
epoch_3 <- checkpoint-123 504 tensores
[
 {
  "epoch": 1,
  "step": 41,
  "loss": 0.2370477318763733
 },
 {
  "epoch": 2,
  "step": 82,
  "loss": 0.2442624568939209
 },
 {
  "epoch": 3,
  "step": 123,
  "loss": 0.25448501110076904
 }
] 
melhor época: 1


In [8]:
# Só no ensaio: a reserva em float16 cabe na GPU? E quanto tempo o treino real deve levar?
if REHEARSAL:
    tokens_per_s = TOKEN_STATS["train"]["total"] * EPOCHS / DURATION_S
    METRICS["rehearsal"] = {"tokens_per_s": round(tokens_per_s, 1),
                            "estimated_min_400_examples_1400_tokens_3_epochs": round(400 * 1400 * 3 / tokens_per_s / 60)}
    if HAS_GPU and USE_4BIT:
        del trainer, model, final_model
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            model16 = load_model(False)
            trainer16 = SFTTrainer(model=model16, args=make_args(os.path.join(WORK_DIR, "teste_fp16"), 1, 1, max_steps=2, evaluate=False),
                                   train_dataset=Dataset.from_list(train_rows[:1]), processing_class=tokenizer, peft_config=LORA_CONFIG)
            trainer16.train()
            METRICS["rehearsal"]["fp16_peak_memory_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        except Exception as error:
            METRICS["rehearsal"]["fp16_error"] = repr(error)[:500]
    with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(METRICS, f, ensure_ascii=False, indent=1)
    print(json.dumps(METRICS["rehearsal"], indent=1))

In [9]:
# Download do zip (e cópia no Drive no modo "train")
zip_path = shutil.make_archive(os.path.join(WORK_DIR, "adaptador_luz_segura" + ("_ensaio" if REHEARSAL else "")), "zip", OUTPUT_DIR)
print(zip_path, f"{os.path.getsize(zip_path) / 2**20:.0f} MiB")
if IN_COLAB:
    if USE_DRIVE and not REHEARSAL:
        shutil.copy(zip_path, PERSIST_DIR)
        from google.colab import drive
        drive.flush_and_unmount()
    from google.colab import files as colab_files
    colab_files.download(zip_path)

/content/adaptador_luz_segura.zip 173 MiB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>